# EDA - Klasifikasi Penggunaan Masker pada Wajah

Notebook ini diperluas agar sesuai dengan proposal: tidak hanya menghitung distribusi kelas, tetapi juga memeriksa kualitas citra, integritas data, outlier, duplikasi, dan verifikasi ROI wajah secara otomatis.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageStat, UnidentifiedImageError

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_SPLITS = ["Train", "Validation", "Test"]
EXPECTED_LABELS = ["WithMask", "WithMaskIncorrect", "WithoutMask"]


In [ ]:
def find_dataset_dir():
    env_dir = os.environ.get("MASK_DATA_DIR")
    if env_dir:
        candidate = Path(env_dir).expanduser()
        if candidate.exists():
            return candidate

    candidates = [
        Path("data/raw"),
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd().parent.parent / "data/raw",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    return Path(os.environ.get("MASK_DATA_DIR", "data/raw")).expanduser()


def image_stats(path: Path) -> dict:
    with Image.open(path) as img:
        rgb = img.convert("RGB")
        gray = img.convert("L")
        width, height = img.size
        rgb_stat = ImageStat.Stat(rgb)
        gray_stat = ImageStat.Stat(gray)

    return {
        "width": width,
        "height": height,
        "aspect_ratio": width / height,
        "file_size_kb": path.stat().st_size / 1024,
        "mean_intensity": gray_stat.mean[0],
        "std_intensity": gray_stat.stddev[0],
        "r_mean": rgb_stat.mean[0],
        "g_mean": rgb_stat.mean[1],
        "b_mean": rgb_stat.mean[2],
    }


def file_sha1(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.sha1()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def sample_paths(frame: pd.DataFrame, split: str, label: str, n: int = 3) -> pd.DataFrame:
    subset = frame[(frame["split"] == split) & (frame["label"] == label)]
    if subset.empty:
        return subset
    return subset.sample(n=min(n, len(subset)), random_state=RANDOM_SEED)


def show_sample_grid(frame: pd.DataFrame, split: str, n: int = 3):
    labels = EXPECTED_LABELS
    fig, axes = plt.subplots(len(labels), n, figsize=(4 * n, 4 * len(labels)))
    if len(labels) == 1:
        axes = np.array([axes])

    for i, label in enumerate(labels):
        subset = sample_paths(frame, split, label, n)
        for j in range(n):
            ax = axes[i, j]
            ax.axis("off")
            if j < len(subset):
                row = subset.iloc[j]
                ax.imshow(Image.open(row["path"]))
                ax.set_title(label + "\n" + row["filename"], fontsize=9)
    plt.suptitle(f"Contoh sampel {split}", y=1.01, fontsize=14)
    plt.tight_layout()
    plt.show()


## 2. Scan Dataset dan Metadata Dasar

In [ ]:
DATA_DIR = find_dataset_dir()
print("Dataset path:", DATA_DIR.resolve())
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan di {DATA_DIR}")

records = []
corrupt_files = []

for split in EXPECTED_SPLITS:
    split_dir = DATA_DIR / split
    if not split_dir.exists():
        continue

    for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
        for path in sorted(class_dir.iterdir()):
            if path.suffix.lower() not in IMAGE_EXTENSIONS:
                continue
            try:
                stats = image_stats(path)
                records.append({
                    "split": split,
                    "label": class_dir.name,
                    "filename": path.name,
                    "path": str(path),
                    **stats,
                })
            except (UnidentifiedImageError, OSError, ValueError) as exc:
                corrupt_files.append({"path": str(path), "error": repr(exc)})

files_df = pd.DataFrame(records)
print("Total gambar:", len(files_df))
print("File korup/tidak terbaca:", len(corrupt_files))
files_df.head()


## 3. Distribusi Split dan Kelas

Bagian ini sesuai dengan proposal poin *eksplorasi distribusi kelas per split* dan membantu melihat apakah ada ketimpangan kelas sejak awal.

In [ ]:
count_table = pd.crosstab(files_df["split"], files_df["label"]).reindex(index=EXPECTED_SPLITS, columns=EXPECTED_LABELS)
count_table


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

class_totals = files_df["label"].value_counts().reindex(EXPECTED_LABELS)
axes[0].bar(class_totals.index, class_totals.values, color=["#4C78A8", "#F58518", "#54A24B"])
axes[0].set_title("Total gambar per kelas")
axes[0].set_xlabel("Kelas")
axes[0].set_ylabel("Jumlah gambar")
axes[0].tick_params(axis="x", rotation=15)

im = axes[1].imshow(count_table.values, cmap="Blues")
axes[1].set_xticks(range(len(EXPECTED_LABELS)), EXPECTED_LABELS, rotation=15)
axes[1].set_yticks(range(len(EXPECTED_SPLITS)), EXPECTED_SPLITS)
axes[1].set_title("Heatmap split x kelas")
for i in range(count_table.shape[0]):
    for j in range(count_table.shape[1]):
        axes[1].text(j, i, int(count_table.iloc[i, j]), ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()


### Ringkasan cepat

- Total gambar: **14.786**
- File korup: **0**
- Split bawaan sudah siap dipakai
- `WithMaskIncorrect` memang lebih sedikit daripada `WithMask` dan `WithoutMask`, jadi F1-score per kelas nanti lebih informatif daripada accuracy saja

## 4. Analisis Resolusi, Ukuran, dan Pencahayaan

Bagian ini mengikuti proposal poin *analisis resolusi, ukuran, dan variasi pencahayaan tiap kelas*.

In [ ]:
files_df[["width", "height", "aspect_ratio", "file_size_kb", "mean_intensity", "std_intensity"]].describe().round(2)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
cols = ["width", "height", "aspect_ratio", "file_size_kb", "mean_intensity", "std_intensity"]
colors = ["#4C78A8", "#F58518", "#54A24B", "#B279A2", "#E45756", "#72B7B2"]
for ax, col, color in zip(axes.flatten(), cols, colors):
    ax.hist(files_df[col], bins=30, color=color, alpha=0.85)
    ax.set_title(col)
    ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()


In [ ]:
brightness_by_class = files_df.groupby("label")[["mean_intensity", "std_intensity", "r_mean", "g_mean", "b_mean"]].agg(["mean", "std", "min", "max"]).round(2)
brightness_by_class


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ["mean_intensity", "std_intensity", "aspect_ratio"], ["#4C78A8", "#F58518", "#54A24B"]):
    grouped = [files_df.loc[files_df["label"] == label, col] for label in EXPECTED_LABELS]
    ax.boxplot(grouped, tick_labels=EXPECTED_LABELS, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.35), medianprops=dict(color="black"))
    ax.set_title(f"{col} per kelas")
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()


### Temuan metadata

- Semua gambar terbaca dengan rasio aspek **1:1**, jadi resize ke ukuran tetap tidak akan merusak proporsi terlalu banyak.
- Resolusi paling dominan adalah **224×224** dan **128×128**.
- Kelas `WithMaskIncorrect` punya rata-rata brightness jauh lebih rendah daripada dua kelas lain. Ini kandidat bias yang harus diingat saat training.
- Ada **19** gambar dengan mean intensity = 0 yang layak ditandai sebagai outlier atau hasil augmentasi ekstrem.

## 5. Visualisasi Sampel per Split

Bagian ini memenuhi proposal poin *visualisasi sampel citra per kelas* sekaligus membantu inspeksi manual label dan ROI.

In [ ]:
for split in EXPECTED_SPLITS:
    show_sample_grid(files_df, split, n=3)


## 6. Audit Integritas Data: File Sangat Gelap dan Duplikasi

Proposal meminta pemeriksaan integritas data. Di sini saya cek dua hal: outlier yang sangat gelap dan duplicate content berdasarkan hash file.

In [ ]:
darkest = files_df.nsmallest(12, "mean_intensity").copy()
darkest[["split", "label", "filename", "mean_intensity", "std_intensity", "width", "height"]]


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, (_, row) in zip(axes.flatten(), darkest.iterrows()):
    ax.imshow(Image.open(row["path"]))
    ax.set_title(row["label"] + "\nmean=" + f"{row['mean_intensity']:.1f}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
hash_df = files_df[["split", "label", "filename", "path"]].copy()
hash_df["sha1"] = [file_sha1(Path(path)) for path in hash_df["path"]]
duplicate_df = hash_df[hash_df.duplicated("sha1", keep=False)].sort_values("sha1")
print("Jumlah baris yang termasuk duplicate exact-match:", len(duplicate_df))
print("Jumlah grup duplicate exact-match:", duplicate_df["sha1"].nunique())
duplicate_df.head(15)


### Catatan integritas

Hasil hash menunjukkan ada exact duplicate antar file. Ini penting karena duplicate lintas split bisa membuat evaluasi model terlalu optimistis. Jadi sebelum training final, duplicate lintas split sebaiknya ditangani atau setidaknya didokumentasikan.

## 7. Audit ROI Wajah Otomatis (Haar Cascade)

Proposal menyebut verifikasi wajah otomatis dengan Haar Cascade atau MTCNN. Untuk EDA, saya pakai **audit cepat berbasis sampel terstratifikasi** dengan Haar Cascade. Hasil ini jangan dianggap ground truth, tetapi cukup untuk melihat apakah pendekatan verifikasi otomatis sederhana cukup andal pada gambar bermasker.

In [ ]:
try:
    import cv2
    cv2_available = True
except ImportError:
    cv2_available = False
    print("cv2 tidak tersedia, audit ROI otomatis dilewati.")

if cv2_available:
    sample_parts = []
    for split in EXPECTED_SPLITS:
        for label in EXPECTED_LABELS:
            subset = files_df[(files_df["split"] == split) & (files_df["label"] == label)]
            sample_parts.append(subset.sample(n=min(100, len(subset)), random_state=RANDOM_SEED))
    roi_audit_df = pd.concat(sample_parts, ignore_index=True)

    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
    detections = []
    for _, row in roi_audit_df.iterrows():
        img = cv2.imread(row["path"])
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
        detections.append({
            "split": row["split"],
            "label": row["label"],
            "face_count": len(faces),
            "face_detected": int(len(faces) > 0),
        })

    roi_result_df = pd.DataFrame(detections)
    roi_summary = roi_result_df.groupby(["split", "label"])["face_detected"].agg(["mean", "sum", "count"]).round(3)
    roi_summary


### Temuan ROI audit

Pada audit cepat ini, Haar Cascade mendeteksi wajah dengan baik pada kelas `WithoutMask`, tetapi jauh lebih lemah pada `WithMask` dan terutama gagal pada `WithMaskIncorrect`. Artinya, untuk dataset wajah bermasker, Haar Cascade kurang andal sebagai validator ROI tunggal. Jika audit ROI ingin dibawa lebih jauh, **MTCNN atau detector modern lain** lebih layak dipakai daripada Haar klasik.

## 8. Kesimpulan EDA

1. EDA ini sudah sesuai dengan butir proposal untuk distribusi kelas, visualisasi sampel, resolusi/ukuran, variasi pencahayaan, dan verifikasi integritas dasar.
2. Dataset cukup rapi untuk dipakai, tetapi ada dua risiko nyata: **exact duplicate** dan **outlier sangat gelap**.
3. ROI wajah secara manual terlihat cukup konsisten, tetapi audit otomatis dengan Haar Cascade menunjukkan keterbatasan besar pada wajah bermasker.
4. Tahap berikut yang paling masuk akal adalah preprocessing terkontrol, enhancement yang benar-benar dibandingkan terhadap baseline, lalu feature extraction/CNN sesuai proposal.